# Tutorial 05: Hardware-Aware Fencing (Stopping VRAM Bleed)

Welcome to the most critical notebook for deploying local AI. In previous tutorials, we built a beautiful, self-correcting CRAG state machine. However, if you try to deploy that system on a consumer GPU (like an RTX 4060 with 8GB of VRAM) using standard library defaults, it will likely crash with an `Out of Memory (OOM)` error within minutes.

In this notebook, we will explore **Asymmetric Hardware Fencing**. We will uncover exactly why naive RAG melts consumer hardware and how to explicitly route specific workloads to the CPU and RAM to keep your GPU completely dedicated to generative inference.

### Prerequisites
* Completion of the previous tutorials.
* Basic understanding of how machine learning models utilize memory (VRAM vs. System RAM).
* *Note: You do not need a GPU to run this notebook, as we will be explicitly simulating CPU offloading.*


### Step 1: The Anatomy of a VRAM Crash

When you build a standard Corrective RAG pipeline, you are actually running three completely separate neural networks simultaneously:

1. **The Embedding Model:** Converts user queries into vectors (e.g., `nomic-embed-text` ~ 1GB VRAM).
2. **The Cross-Encoder Grader/Reranker:** Evaluates document relevance (e.g., `bge-reranker` ~ 2GB VRAM).
3. **The Generative LLM:** Writes the final answer (e.g., `Llama-3-8B-Instruct` ~ 5.5GB VRAM in 4-bit quantization).

**The Naive Approach (The Broken Way):**
Most libraries default to `device="cuda"` or `device="auto"` for everything.
$1GB + 2GB + 5.5GB = 8.5GB$.
If you have an 8GB GPU, the moment your CRAG loop hits the LLM generation step, CUDA runs out of memory, triggers system swap, and your generation drops from 40 tokens/second to 0.5 tokens/second—or the process just gets killed.

---

### Step 2: Implementing the Asymmetric Fence

To fix this, we implement a strict "fence."

* **GPU:** Strictly reserved for the Generative LLM. (It needs the VRAM bandwidth the most).
* **CPU & System RAM:** Assigned to handle Vector Storage, Embeddings, and Reranking. (These tasks are highly parallelizable and run surprisingly fast on modern CPUs).

Let's build a hardware-aware embedding class that refuses to touch the GPU. We will use the `fastembed` library, which is heavily optimized for CPU execution.


In [ ]:
# The Hardware-Aware Approach
from fastembed import TextEmbedding
import psutil

print("--- Initializing Hardware-Fenced Embedding Model ---")

# 1. We explicitly force the execution provider to the CPU.
# We also limit the threads. Why? If you have a 16-core machine, 
# ONNX runtime will try to use all 16 cores for embeddings. 
# This causes "CPU Starvation" and freezes your database and OS. 
# We fence it to 4 threads.
cpu_fenced_embeddings = TextEmbedding(
    model_name="BAAI/bge-small-en-v1.5",
    providers=["CPUExecutionProvider"], # Strictly forbid CUDA
    threads=4                           # Prevent CPU Starvation
)

# 2. Test the fenced execution
query = "Explain asymmetric resource allocation."
vector = list(cpu_fenced_embeddings.embed([query]))[0]

print(f"✅ Embedded query into vector of size {len(vector)}.")
print(f"✅ Executed purely on CPU/RAM without consuming a single megabyte of VRAM.")


### Step 3: Fencing the Generative LLM (Ollama)

Now that our embeddings are safely confined to the CPU, we can confidently allocate our GPU entirely to Ollama for the generative tasks (like our Router, Grader, and final Generator nodes).

When running your system locally via Docker or bare-metal, you control this via context parameters.


In [ ]:
from langchain_community.llms import Ollama

print("--- Initializing GPU-Bound Generative LLM ---")

# 1. We allocate the LLM to Ollama. 
# By keeping num_ctx (context window) tightly controlled, we ensure
# the KV Cache doesn't expand and accidentally overflow our remaining VRAM.
fenced_llm = Ollama(
    model="llama3",
    temperature=0,
    num_ctx=4096 # Cap the memory footprint of the context window
)

response = fenced_llm.invoke("Say 'GPU isolation confirmed.'")
print(f"🤖 LLM Check: {response}")


### The Architectural Win

By manually defining where each piece of the pipeline lives:

1. **Zero-VRAM Bleed:** Your GPU never crashes. You can confidently deploy this on a consumer RTX 4060 or a low-tier cloud instance (like an AWS `g4dn.xlarge`).
2. **Concurrency:** While the LLM is busy generating text on the GPU for User A, the CPU is simultaneously embedding a query for User B. You have achieved true hardware parallelism.
3. **Cost Efficiency:** You no longer need to pay for 24GB enterprise GPUs (like an A10G or A100) just to run a basic RAG pipeline.

*Note: In this repository, you will see this asymmetric fencing strategy applied natively via [`docker-compose.yaml`](../docker-compose.yaml) and [`scripts/factory.py`](../scripts/factory.py).*

---

### 🏋️ Challenge

Currently, we hardcoded `providers=["CPUExecutionProvider"]`. But what if someone clones your repository and actually *does* have a massive 24GB RTX 4090? They would want to use the GPU for everything!

**Your Task:**
Write a Python utility function called `get_optimal_execution_provider()`. Use the `torch` or `pynvml` library to check the host machine's total VRAM:

* If Total VRAM > 16GB, return `["CUDAExecutionProvider", "CPUExecutionProvider"]`.
* If Total VRAM <= 16GB, return `["CPUExecutionProvider"]`.

This is called **Graceful Degradation**, and it allows your application to run perfectly on both consumer laptops and enterprise cloud clusters without altering the code.

In our final tutorial, **Tutorial 06: Production Hardening**, we will take everything we have built and learn how to containerize it, measure its accuracy offline with Ragas, and build automated CI/CD quality gates.
